In [14]:
from typing import Literal, Set
import numpy as np

rng = np.random.default_rng(42)

CellState = Literal["edge", "center"]
CellAction = Literal["death", "birth", "skip"]

class Cell:
    def __init__(self, cell_id: int, parent_id: int,
                 state: CellState,
                 x: int, y: int,
                 death_rate: float, birth_rate: float,
                 mutations: Set[int]):
        self.cell_id = cell_id
        self.parent_id = parent_id
        self.state = state
        self.x, self.y = x, y
        self.death_rate = death_rate
        self.birth_rate = birth_rate
        self.mutations = mutations

    def choose_action(self) -> CellAction:
        """Choose an action based on cells birth, death probability. returns "birth", "death", or "skip". returning "birth"
        does not mean the cell multiplies, it may be in a state (surrounded state) where it can't reproduce (under certain models)"""
        if rng.random() < self.death_rate:
            return "death"
        elif rng.random() < self.birth_rate:
            return "birth"
        return "skip"

class World:
    def __init__(self, size: int,
                 time_step_hours: float = 2.0,
                 target_cells: int = 1000,
                 growth_model: Literal["boundary", "unrestricted"] = "boundary",
                 mutation_rate: float = 1.0):

        #check if inputs make sense
        if size < 1 or target_cells < 1:
            raise ValueError("size and target_cells must be positive")

        if mutation_rate < 0 or mutation_rate > 1:
            raise ValueError(f"`mutation_rate` must be a float between [0, 1], but {mutation_rate} was given")

        if growth_model not in ("boundary", "unrestricted"):
            raise ValueError("growth model must be either boundary or unrestricted")        

        #if the matrix is not enuf to even reach target_cells
        if target_cells > size * size:
            raise ValueError("the matrix has too few positions compared to the upper limit set by target_cells")


        self.cell_count = 0
        self.size = size
        self.grid = np.full((size, size), None, dtype=Cell)

        # each cell is given a unique identifying ID, that starts from 1 and goes up
        self.next_id = 1

        #to track sim time --> can print out for extra detail if needed/want
        self.step = 0
        self.time_step_hours = time_step_hours

        #to know when to stop sim (default being 1000 like in the paper)
        self.target_cells = target_cells

        #type of model to use --> boundary means don't push to make space when the 8 surrounding cells are filled, unrestricted means do that
        self.growth_model = growth_model

        # mutations follows a Jukes-Cantor Model (equal nucleotide frequency, independent mutation rate at each position, equal substitution rates between any 2 nucleotides)
        # and an infinite site assumption (a specific base is only ever mutated once -> one strain cannot mutate a base twice AND 2 strains cannot independently mutate the same base
        # to implement infinite site assumption, whenever we create a mutation, we increase the size of the genome by 1, and mutate it (so it's impossible to mutate a previously added base)
        # mutations are stored in a set of bases i.e. {1, 5, 8} = this individual has a mutation in position 1, 5, 8
        #
        # next base position to add
        self.next_mutation_position = 1
        # probability of offspring to get a mutation (1.0 in the paper)
        self.mutation_rate = mutation_rate

        #mutation position --> (ancestral nucleotide, mutated nucleotide)
        #in this format after running stuff:
        #{1: ("A","G"), 2: ("C","T"), 3: ("G","A")}
        self.mutation_sites = {}
        



    def update(self):
        #after adding remove cell function (which happens when cell action is death), sometimes sim keeps running when there's already nothing
        #repetitive bc when there's nothing, the next step is also going to be nothing
        #stop returning if the above case or if it goes past the set upper limit on # of cells
        if self.cell_count >= self.target_cells or self.cell_count == 0:
            return

        # Every cell that exists at the start of update should be updated, in random order to not be biased
        cells = self.grid.flatten()
        cells = cells[cells != None]
        rng.shuffle(cells)

        for cell in cells:
            action = cell.choose_action()

            match action:
                case "death":
                    self.remove_cell(cell)
                case "birth":
                    self.birth_new_cell(cell)
            # print(f"cell {cell.cell_id} chooses to {action}")

            if self.cell_count >= self.target_cells:
                break
        
        #update self.step and self.time_hours accordingly
        self.step += 1


    def _set_cell_at_position(self, cell: Cell):
        self.grid[cell.x,cell.y] = cell
        self.cell_count+=1

    def _get_next_cell_id(self):
        """return next unused id number and increment counter"""
        self.next_id += 1
        return self.next_id - 1

    def birth_new_cell(self, parent: Cell):
        """Look for a empty neighboring position from the parent cell and create a new Cell there"""
        empty_neighboring_indices = []
        for x in range(parent.x - 1, parent.x + 2):
            for y in range(parent.y - 1, parent.y + 2):
                if x == parent.x and y == parent.y:
                    continue
                if x < 0 or y < 0 or x >= self.size or y >= self.size:
                    continue
                if self.grid[x,y] is None:
                    empty_neighboring_indices.append((x,y))


        if empty_neighboring_indices:
            cell_position = empty_neighboring_indices[rng.integers(len(empty_neighboring_indices))]
        elif self.growth_model == "boundary":
            #if boundary, u can't do anything abt it
            return
        else:
            #if unrestricted, make space by pushing surroundings
            cell_position = self.push_to_make_space(parent)
            
            if cell_position is None:
                return

        # mutations are inherited from the parent, and if cell mutates in process, add a new mutation
        mutations = parent.mutations.copy()
        if rng.random() < self.mutation_rate:
            #equal prob of each ancestral nucleotide
            ancestral = str(rng.choice(["A", "C", "G", "T"]))

            #equal prob of each of other three nucleotides
            alternatives = [base for base in ["A", "C", "G", "T"] if base != ancestral]
            mutated = str(rng.choice(alternatives))

            self.mutation_sites[self.next_mutation_position] = (ancestral, mutated)

            mutations.add(self.next_mutation_position)
            self.next_mutation_position += 1


        self._set_cell_at_position(Cell(
            cell_id = self._get_next_cell_id(), parent_id = parent.cell_id,
            state ="edge",
            x =cell_position[0], y = cell_position[1],
            death_rate = parent.death_rate, birth_rate = parent.birth_rate,
            mutations = mutations
        ))


    def remove_cell(self, cell:Cell):
        self.grid[cell.x, cell.y] = None
        self.cell_count -= 1

    def push_to_make_space(self, parent: Cell) -> tuple[int, int] | None:
        #output of None meaning it failed to make space --> reached matrix limits
        #8 lattice directions, including diagonals
        directions = [
            (dx, dy)
            for dx in (-1, 0, 1)
            for dy in (-1, 0, 1)
            if (dx, dy) != (0, 0)
        ]

        dx, dy = directions[rng.integers(len(directions))]

        x = parent.x + dx
        y = parent.y + dy

        occupied_positions = []

        #find first empty position in chosen direction
        while True:
            if x < 0 or y < 0 or x >= self.size or y >= self.size:
                # raise RuntimeError("pushing reached grid boundary, u can try again w/ larger grid inputs")
                return None

            if self.grid[x, y] is None:
                break

            occupied_positions.append((x, y))

            x += dx
            y += dy
        
        #do in reverse to move farthest cell first so that everything moves cleanly/no overlap
        for old_x, old_y in reversed(occupied_positions):
            cell = self.grid[old_x, old_y]

            new_x = old_x + dx
            new_y = old_y + dy

            self.grid[new_x, new_y] = cell
            self.grid[old_x, old_y] = None
            
            cell.x = new_x
            cell.y = new_y
        
        #this neighboring position now empty and birth_new_cell() will put the cell in this empty slot
        return parent.x + dx, parent.y + dy



# Helper Functions
def visualize_world(world: World):
    """hacky way to visualize each world grid"""
    grid = world.grid
    for row in grid:
        print("  ".join(["..." if x is None else f"{x.cell_id:03d}" for x in row]), flush=True)


def build_sequence(world: World, cell: Cell) -> str:
    '''build and return dna sequence for one cell --> world has shared mutation-site records, cell contains mutation positions carried by specific cell'''
    sequence = []

    #visit sites in numerical order
    #using same order for every cell keeps sequences comparable
    for position in sorted(world.mutation_sites):
    
        #retrieve og and changed nucleotide at this site
        #the value of dictionary is a tuple with (ancestral, mutated)
        ancestral, mutated = world.mutation_sites[position]

        #check if cell has mutation at this site --> yes means add mutated nucleotide to seq, no means add og nucleotide
        if position in cell.mutations:
            sequence.append(mutated)
        else:
            sequence.append(ancestral)

    return "".join(sequence)


def convert_fasta(world: World, filename: str):
    '''converts all the cell sequences in fasta format'''
    living_cells = [cell for cell in world.grid.flatten() if cell is not None]

    if not living_cells:
        raise ValueError("there are no living cells to export")
    
    if not world.mutation_sites:
        #stop bc build_sequence would produce empty sequences
        raise ValueError("no mutation sites were generated, so there are no nonempty sequences to export")

    #sort cells by ID in increasing order
    living_cells.sort(key = lambda cell: cell.cell_id)

    #creates file if needed/overwrites it if alr exists
    with open(filename, "w") as fasta:
        #one FASTA entry for each living cell
        for cell in living_cells:
            #build cell's dna sequence using its mutation positions
            sequence = build_sequence(world, cell)

            #fasta file formatting
            fasta.write(f">cell_{cell.cell_id}\n")
            fasta.write(f"{sequence}\n")

In [15]:
from IPython.display import clear_output
from time import sleep

#reusable for diff inputs
world = World(size = 20, time_step_hours = 2, target_cells = 100, growth_model = "unrestricted")
center = world.size // 2
world._set_cell_at_position(
    Cell(0, -1, "edge", center, center, 0.2, 0.9,
         mutations = set())
)

# world._set_cell_at_position(
#     Cell(0, -1, "edge", 7, 7, .5, .5)
# )

for i in range(15):
    world.update()
    clear_output(wait = True)

    print("Updates completed:", world.step)
    print("Simulated hours:", world.step * world.time_step_hours)
    print("Living cells:", world.cell_count)
    print()

    visualize_world(world)
    print("\n")
    
    if world.cell_count == 0 or world.cell_count >= world.target_cells:
        break

    sleep(1)

print("*" * 10)
visualize_world(world)

Updates completed: 10
Simulated hours: 20
Living cells: 100

...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  ...  ...  ...  102  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  ...  ...  ...  047  104  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  ...  ...  ...  ...  035  096  ...  088  ...  ...  ...  089  ...  ...  ...
...  ...  ...  ...  ...  068  090  ...  063  072  017  099  ...  ...  ...  066  116  ...  ...  ...
...  ...  ...  ...  ...  ...  055  043  073  086  000  124  121  106  028  120  ...  ...  ...  ...
...  ...  ...  ...  ...  ...  ...  023  097  107  ...  010  059  095  083  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  115  ...  058  078  050

In [18]:
#testing build_sequence

for cell in world.grid.flatten():
    if cell is not None:
        dna = build_sequence(world, cell)

        print("Cell ID:", cell.cell_id)
        print("Mutation positions:", sorted(cell.mutations))
        print("DNA sequence:", dna)
        print()

Cell ID: 102
Mutation positions: [4, 10, 47, 102]
DNA sequence: TTAACCAATCCTCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGTTGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTGCCATGAGACCTGATGGAAAGCG

Cell ID: 47
Mutation positions: [4, 10, 47]
DNA sequence: TTAACCAATCCTCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGTTGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG

Cell ID: 104
Mutation positions: [4, 10, 17, 35, 104]
DNA sequence: TTAACCAATCCTCTATTCCAAGGGTCCAACACGTTTGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCGATGAGACCTGATGGAAAGCG

Cell ID: 35
Mutation positions: [4, 10, 17, 35]
DNA sequence: TTAACCAATCCTCTATTCCAAGGGTCCAACACGTTTGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG

Cell ID: 96
Mutation positions: [4, 10, 17, 96]
DNA sequence: TTAACCAATCCTCTATTCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAACTGTCTTCCATGAGACCTGATGGAAAGCG

Cell ID: 88
Mutation positions: [4, 10, 17, 59, 88]
DNA 

In [19]:
#testing convert_fasta

convert_fasta(world, "test_living_cells.fasta")

with open("test_living_cells.fasta", "r") as fasta:
    contents = fasta.read()

print(contents)

>cell_0
TTATCCAATGCTCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG
>cell_2
AGATCCAATGCTCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG
>cell_3
AGGTCCAATGCTCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG
>cell_10
TTAACCAATCCTCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG
>cell_11
TTATCCAATGTTCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG
>cell_12
AGGTCAAATGCCCTATCCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG
>cell_13
AGATCCAATGCTGTATCCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTATATACTATCACAGCGATAGGGACTCGTTAACCATCGTAATTGTCTTCCATGAGACCTGATGGAAAGCG
>cell_16
TTAACCTATGCTCTAACCCAAGGGTCCAACACGTATGGTAGTCCCGATGTCCTCTA

In [ ]:
for i in range(3):
    clear_output(wait=False)
    print(f"Frame {i}", flush=True)
    sleep(0.4)